In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataCleaning").getOrCreate()

df = spark.read.csv(
    "customers_data.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

In [ ]:
from pyspark.sql.functions import col, explode
from pyspark.sql.types import StructType, ArrayType

def flatten_df(df):
    
    def _flatten(schema, prefix=""):
        fields = []
        for field in schema.fields:
            name = f"{prefix}{field.name}"
            dtype = field.dataType
            if isinstance(dtype, StructType):
                # recurse into struct
                fields.extend(_flatten(dtype, prefix=name + "."))
            elif isinstance(dtype, ArrayType) and isinstance(dtype.elementType, StructType):
                # explode array of structs
                fields.append((name, "explode"))
            else:
                fields.append((name, "column"))
        return fields

    flat_fields = _flatten(df.schema)

    # Explode arrays first
    for f, t in flat_fields:
        if t == "explode":
            df = df.withColumn(f, explode(col(f)))

    # Build select expressions
    select_exprs = []
    for f, t in flat_fields:
        if t == "column":
            select_exprs.append(col(f).alias(f.replace(".", "_")))
        elif t == "explode":
            # flatten struct fields inside exploded array
            struct_fields = df.select(col(f + ".*")).schema.fields
            for sf in struct_fields:
                select_exprs.append(col(f + "." + sf.name).alias(f.replace(".", "_") + "_" + sf.name))

    return df.select(*select_exprs)


In [ ]:
nested_df = spark.read.json("nested_orders.json")

flat_df = flatten_df(nested_df)
flat_df.printSchema()
flat_df.show(truncate=False)
